# Data Gathering

In [1]:
import pandas as pd
df = pd.read_csv('twitter_training.csv')

# Data Exploration

In [2]:
df

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype
---  ------                                                 --------------  -----
 0   2401                                                   74681 non-null  int64
 1   Borderlands                                            74681 non-null  str  
 2   Positive                                               74681 non-null  str  
 3   im getting on borderlands and i will murder you all ,  73995 non-null  str  
dtypes: int64(1), str(3)
memory usage: 11.4 MB


In [4]:
df.isna().sum()

2401                                                       0
Borderlands                                                0
Positive                                                   0
im getting on borderlands and i will murder you all ,    686
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(2700)

In [6]:
df['Positive'].value_counts()

Positive
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

# Data Cleaning

In [7]:
df= df[df['Positive']!='Irrelevant']

In [8]:
df.drop(columns=['2401','Borderlands'],inplace=True)

In [9]:
df.rename(columns={'Positive':'Sentiment','im getting on borderlands and i will murder you all ,':'Tweets'},inplace=True)

In [10]:
df.dropna(inplace=True)

In [11]:
df.drop_duplicates(inplace=True)

In [12]:
df

,Sentiment,Tweets
0,Positive,I am coming to the borders and I will kill you...
1,Positive,im getting on borderlands and i will kill you ...
2,Positive,im coming on borderlands and i will murder you...
3,Positive,im getting on borderlands 2 and i will murder ...
4,Positive,im getting into borderlands and i can murder y...
...,...,...
74676,Positive,Just realized that the Windows partition of my...
74677,Positive,Just realized that my Mac window partition is ...
74678,Positive,Just realized the windows partition of my Mac ...
74679,Positive,Just realized between the windows partition of...


# Feature Encoding

In [13]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Sentiment'] = le.fit_transform(df['Sentiment'])

In [34]:
print(le.classes_)

['Negative' 'Neutral' 'Positive']


In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [15]:
tok = Tokenizer(oov_token='Nothing')
tok.fit_on_texts(df['Tweets'])

In [16]:
vocab_size = len(tok.word_index)+1 #Vocabulary Size

In [17]:
tok.word_counts

OrderedDict([('i', 24307),
             ('am', 1516),
             ('coming', 403),
             ('to', 23272),
             ('the', 35286),
             ('borders', 11),
             ('and', 21550),
             ('will', 2674),
             ('kill', 348),
             ('you', 9290),
             ('all', 4490),
             ('im', 547),
             ('getting', 1046),
             ('on', 9736),
             ('borderlands', 1412),
             ('murder', 67),
             ('2', 3709),
             ('me', 5514),
             ('into', 947),
             ('can', 2891),
             ('so', 6395),
             ('spent', 179),
             ('a', 19027),
             ('few', 467),
             ('hours', 481),
             ('making', 527),
             ('something', 752),
             ('for', 12481),
             ('fun', 1329),
             ('if', 2855),
             ("don't", 1337),
             ('know', 1391),
             ('huge', 310),
             ('fan', 228),
             ('maya', 22),
 

In [18]:
sequences = tok.texts_to_sequences(df['Tweets'])

In [19]:
from keras.utils import pad_sequences
sequences = pad_sequences(sequences,padding='pre') #maxlen=1000 is also a parameter

In [20]:
sequences 

array([[   0,    0,    0, ...,  452,   15,   28],
       [   0,    0,    0, ...,  452,   15,   28],
       [   0,    0,    0, ..., 1732,   15,   28],
       ...,
       [   0,    0,    0, ...,  152, 1020, 2536],
       [   0,    0,    0, ...,   78,   41, 2536],
       [   0,    0,    0, ...,    3, 1020, 2536]],
      shape=(57484, 166), dtype=int32)

In [21]:
from tensorflow.keras.layers import SimpleRNN,Input,Dense,Embedding,Dropout
from tensorflow.keras.models import Sequential

In [22]:
X = sequences

In [23]:
Y = df['Sentiment'].values

In [24]:
Y

array([2, 2, 2, ..., 2, 2, 2], shape=(57484,))

In [25]:
X

array([[   0,    0,    0, ...,  452,   15,   28],
       [   0,    0,    0, ...,  452,   15,   28],
       [   0,    0,    0, ..., 1732,   15,   28],
       ...,
       [   0,    0,    0, ...,  152, 1020, 2536],
       [   0,    0,    0, ...,   78,   41, 2536],
       [   0,    0,    0, ...,    3, 1020, 2536]],
      shape=(57484, 166), dtype=int32)

In [26]:
vocab_size

29230

In [27]:
from sklearn.model_selection import train_test_split

X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [28]:
model = Sequential([Input(shape=(99,)),
                    Embedding(input_dim=vocab_size,output_dim=128,mask_zero=True),
                    SimpleRNN(32),
                    Dropout(0.3),
                    Dense(3,activation='softmax')
                    ])

In [29]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [30]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 99, 128)        │     3,741,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         5,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,746,691 (14.29 MB)

 Trainable params: 3,746,691 (14.29 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

model.fit(X_train,Y_train,validation_data=(X_test,Y_test),epochs=10,callbacks=[early_stop],batch_size=64)

Epoch 1/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 30s 39ms/step - accuracy: 0.7494 - loss: 0.6068 - val_accuracy: 0.8819 - val_loss: 0.3225
Epoch 2/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 30s 42ms/step - accuracy: 0.9355 - loss: 0.1848 - val_accuracy: 0.9022 - val_loss: 0.2683
Epoch 3/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 31s 43ms/step - accuracy: 0.9674 - loss: 0.0918 - val_accuracy: 0.9143 - val_loss: 0.2641
Epoch 4/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 31s 43ms/step - accuracy: 0.9768 - loss: 0.0653 - val_accuracy: 0.9138 - val_loss: 0.2814
Epoch 5/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 30s 42ms/step - accuracy: 0.9811 - loss: 0.0522 - val_accuracy: 0.9111 - val_loss: 0.3055
Epoch 6/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 31s 42ms/step - accuracy: 0.9843 - loss: 0.0442 - val_accuracy: 0.9095 - val_loss: 0.3329


In [32]:
import pickle
with open('tokenizer.pkl','wb') as file:
    pickle.dump(tok,file)

In [33]:
model.save('model.h5')